In [ ]:
# Importing Libraries
import requests
import pandas as pd
from bs4 import BeautifulSoup
import textwrap

In [ ]:
# Defining The Base URL
url = 'https://quotes.toscrape.com/'

# Display A Full List With Every Possible Tag
def get_tags():
  print('Collecting Tags...')

  # Empty Set
  all_tags = set()

  current_path = 'page/1/'

  # Looping Through Pages
  while current_path:

    # Accessing URL
    response = requests.get(url + current_path)
    soup = BeautifulSoup(response.text, 'lxml')

    # Looping Through The Tags Of The Current Page
    page_tags = soup.find_all('a', class_= 'tag')

    for tag in page_tags:

      all_tags.add(tag.text)

    # Next Page
    next_button = soup.find('li', class_= 'next')
    current_path = next_button.find('a')['href'].lstrip('/') if next_button else None

  # Handling Text Width
  text = ', '.join(list(sorted(all_tags)))
  wrapped_text = textwrap.fill(text, width= 80)

  return wrapped_text

# Defining Bio Function
def get_bio(relative_path):

  # Defining The Relative Path That Leads To Author's Bio
  full_url = url + relative_path
  response = requests.get(full_url)
  soup = BeautifulSoup(response.text, 'lxml')

  # Getting Bio & Info From The Relative Path
  full_name = soup.find('h3', class_= 'author-title').text.strip()
  born_date = soup.find('span', class_= 'author-born-date').text.strip()
  born_location = soup.find('span', class_= 'author-born-location').text.strip()
  description = soup.find('div', class_= 'author-description').text.strip()

  # Spliting born_location
  born_location = born_location.replace('in', '').strip()
  parts = [p.strip() for p in born_location.split(',')]

  # Handling Different Cases Of Location Of birth
  if len(parts) >= 3:
    country = parts[-1]
    state = parts[1]
    city = parts[0]
  elif len(parts) == 2:
    country = parts[-1]
    state = 'Unknown'
    city = parts[0]
  else:
    country = parts[0]
    state = 'Unknown'
    city = 'Unknown'


  return full_name, born_date, country, state, city, description


# Defining Scraping Function
def scraping():

  # Displaying Tags
  print('Feel Free To Choose From The Following Tags:\n')
  print(get_tags())
  print('-' * 80 + '\n')

  # Input Data
  user_prefs = input('Enter Tags That Are Related To Your Preferences: ').lower().strip()
  user_tags = [t.strip().replace(',', '') for t in user_prefs.split() if t.strip()]

  # If Wants The Bio Of The Author
  if_bio = input('Do You Want To Take A Look At The Author\'s Bio? (yes/no): ').lower().strip()

  # Empty List
  results = []

  # Looping Through Pages
  current_path = 'page/1/'

  while current_path:

    # Requesting The URl
    response = requests.get(url + current_path)
    soup = BeautifulSoup(response.text, 'lxml')

    # Looping Throught Quotes:
    quote_blocks = soup.find_all('div', class_= 'quote')


    for quote in quote_blocks:

      q_tags = [t.text for t in quote.find_all('a', class_= 'tag')]

      # The Condition & Filtration of preferences

      if any(p in q_tags for p in user_tags):

          # Scraping Data
          quote_text = quote.find('span', class_= 'text').text.strip()
          quote_author = quote.find('small', class_= 'author').text.strip()

          # Looping Into Tags
          quote_tags = []
          for tag in quote.find_all('a', class_= 'tag'):
            quote_tags.append(tag.text)

          quote_tags = ', '.join(quote_tags)

          # Displayig Results
          wrapped_quote = textwrap.fill(quote_text, width= 80)
          print(f'Quote: {wrapped_quote}')
          print(f'Author: {quote_author}')
          print(f'Related Tags: {quote_tags}\n')

          # Default Values Of The Author Info If User Rejected
          full_name, born_date, country, state, city, description = 'N/A', 'N/A', 'N/A', 'N/A', 'N/A', 'N/A'

          # If Wants Bio Of The Author
          if if_bio == 'yes':

            # Defining Relative Path
            relative_url = quote.find('a', string= '(about)')['href']

            # Assigning Values To The get_bio() function
            full_name, born_date, country, state, city, description = get_bio(relative_url)

            # Appending Data To The results List:
            results.append({
                'Full Name': full_name,
                'Quote': quote_text,
                'Related Tags': quote_tags,
                'Born Date': born_date,
                'Country': country,
                'State': state,
                'City': city,
                'Biography': description[:100]
            })

          else:
            results.append({
                'Quote': quote_text,
                'Author': quote_author,
                'Related Tags': quote_tags
            })

    # Moving To The Next Page
    next_button = soup.find('li', class_= 'next')
    current_path = next_button.find('a')['href'].lstrip('/') if next_button else None

  # Saving Data
  df = pd.DataFrame(results)
  df.to_csv('quotes.csv', index= False)

scraping()

Feel Free To Choose From The Following Tags:

abilities, activism, adulthood, adventure, age, alcohol, aliteracy, apathy,
attributed, attributed-no-source, authors, be-yourself, beatles, better-life-
empathy, bilbo, books, change, children, chocolate, choices, christianity,
classic, comedy, connection, contentment, courage, death, deep-thoughts,
difficult, dreamers, dreaming, dreams, drug, dumbledore, edison, education,
elizabeth-bennet, failure, fairy-tales, fairytales, faith, fantasy, fate, fear,
food, friends, friendship, girls, god, good, growing-up, grown-ups, happiness,
hate, heartbreak, hope, humor, imagination, indifference, insanity, inspiration,
inspirational, integrity, jane-austen, journey, knowledge, lack-of-friendship,
lack-of-love, learning, library, lies, life, literature, live, live-death-love,
lost, love, lying, marriage, mind, miracle, miracles, misattributed-eleanor-
roosevelt, misattributed-john-lennon, misattributed-mark-twain, misattributed-
to-c-s-lewis, misattr